## Feature Engineering

feature_engineering.py

### What feature_engineering.py handle ?

* Automatically detects the energy column
* Creates advanced time-based features
* Adds rolling statistical features
* Adds lag features
* Adds deviation (z-score) features
* Cleans NaN rows
* Returns a feature-rich dataset

#### Information :

The feature_engineering() function enriches raw time-series energy data with temporal, rolling statistical, lag, cyclical, and deviation-based features. These engineered features enable unsupervised anomaly detection models to identify contextual anomalies rather than just extreme values.

## Import Libraries 

In [ ]:
import numpy as np
from src.preprocessing import detect_energy_columns

## Function

In [ ]:
def feature_engineering(df):

This function:

* Automatically detects the energy column
* Creates advanced time-based features
* Adds rolling statistical features
* Adds lag features
* Adds deviation (z-score) features
* Cleans NaN rows
* Returns a feature-rich dataset

It transforms raw energy data into a machine learning-ready time-series dataset.

## Detect Energy Column Automatically

In [ ]:
 energy_cols = detect_energy_columns(df)

Instead of hardcoding "electricity" or "energy":

You dynamically detect the energy column.

* Makes your system robust
* Works with different datasets
* More professional

If no column found:

raise ValueError("No energy column detected in dataset.")

Prevents silent model failure.

## Time-Based Features

In [ ]:
  # -----------------------------
    df["hour"] = df["timestamp"].dt.hour
    df["day"] = df["timestamp"].dt.day
    df["month"] = df["timestamp"].dt.month
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["week"] = df["timestamp"].dt.isocalendar().week.astype(int)
    df["quarter"] = df["timestamp"].dt.quarter
    df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)

These extract time components from timestamp.Why?

Energy consumption depends heavily on:

Hour of day

Day of week

Month

Season

Weekend vs weekday

Example:

* AC usage high at 2 PM

* Offices closed on Sunday

* Winter vs Summer consumption differs

These features help model detect seasonality-based anomalies.

## Cyclical Encoding

In [ ]:
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

Time is cyclical:

23:00 → 00:00 are close
But numerically:
23 and 0 look far apart.

Sin/Cos encoding converts hour into circular representation.

This helps ML understand:

* Midnight is near 11 PM
* December is near January

Without this, models misinterpret time relationships.

This is advanced time-series feature engineering.

## Rolling Statistical Features

In [ ]:
    windows = [6, 12, 24, 48, 168]

    for w in windows:
        df[f"roll_mean_{w}"] = df[energy_col].rolling(w).mean()
        df[f"roll_std_{w}"] = df[energy_col].rolling(w).std()
        df[f"roll_min_{w}"] = df[energy_col].rolling(w).min()
        df[f"roll_max_{w}"] = df[energy_col].rolling(w).max()

These represent:

6 hours

12 hours

24 hours (1 day)

48 hours (2 days)

168 hours (1 week)

For each window:

df[f"roll_mean_{w}"]
df[f"roll_std_{w}"]
df[f"roll_min_{w}"]
df[f"roll_max_{w}"]

Rolling mean = moving average
Rolling std = variability
Rolling min/max = local extremes

Why important?

An anomaly is usually:

Energy suddenly higher than normal recent pattern.

Rolling features allow model to compare:

Current value vs recent behavior.

Example:

If normal = 200 units
Suddenly = 500 units

Rolling mean will detect abnormal deviation.

## Lag Feature

In [ ]:
    lags = [1,2,3,6,12,24,48,72,168,336]

    for lag in lags:
        df[f"lag_{lag}"] = df[energy_col].shift(lag)

Lag means:

Previous values.

Example:

df["lag_1"]

* Means energy 1 hour ago.

Why useful?

* Energy depends on previous consumption.

Example:

* If last 5 hours were stable

Suddenly spike occurs
*  Likely anomaly

Lag features give model memory.

## Deviation Features (Z-Score)

In [ ]:
    df["zscore_24"] = (df[energy_col] - df["roll_mean_24"]) / (df["roll_std_24"] + 1e-5)
    df["zscore_168"] = (df[energy_col] - df["roll_mean_168"]) / (df["roll_std_168"] + 1e-5)

Formula:

(Current - Rolling Mean) / Rolling Std

This measures:

How many standard deviations away current value is.

If z-score > 3 → Strong anomaly signal.

This is statistical anomaly detection concept integrated into ML features.

Very powerful.

## Drop NaN Values

In [ ]:
    df = df.dropna()

Rolling and lag create missing values at start.

Example:

If rolling window = 168 hours
First 167 rows become NaN.

So:

df = df.dropna()

Removes incomplete rows.

Clean dataset for ML training.

## Print Summary

In [ ]:
print("✅ Feature Engineering Completed")
    print("Total Features:", df.shape[1])
    print("🔎 Sample Columns:", df.columns[:50])


Shows how many new features created.

Usually increases from:

2–3 columns → 50+ columns

This makes your dataset feature-rich.

### WHole Code :

In [ ]:
import numpy as np
from src.preprocessing import detect_energy_columns

def feature_engineering(df):

    print("⚙️ Feature Engineering...")

    # -----------------------------
    # Detect Energy Column Dynamically
    # -----------------------------
    energy_cols = detect_energy_columns(df)

    if len(energy_cols) == 0:
        raise ValueError("No energy column detected in dataset.")

    energy_col = energy_cols[0]
    print(f"Detected Energy Column: {energy_col}")

    # -----------------------------
    # Time Features
    # -----------------------------
    df["hour"] = df["timestamp"].dt.hour
    df["day"] = df["timestamp"].dt.day
    df["month"] = df["timestamp"].dt.month
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["week"] = df["timestamp"].dt.isocalendar().week.astype(int)
    df["quarter"] = df["timestamp"].dt.quarter
    df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)

    # -----------------------------
    # Cyclical Encoding
    # -----------------------------
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

    # -----------------------------
    # Rolling Features
    # -----------------------------
    windows = [6, 12, 24, 48, 168]

    for w in windows:
        df[f"roll_mean_{w}"] = df[energy_col].rolling(w).mean()
        df[f"roll_std_{w}"] = df[energy_col].rolling(w).std()
        df[f"roll_min_{w}"] = df[energy_col].rolling(w).min()
        df[f"roll_max_{w}"] = df[energy_col].rolling(w).max()

    # -----------------------------
    # Lag Features
    # -----------------------------
    lags = [1,2,3,6,12,24,48,72,168,336]

    for lag in lags:
        df[f"lag_{lag}"] = df[energy_col].shift(lag)

    # -----------------------------
    # Deviation Features
    # -----------------------------
    df["zscore_24"] = (df[energy_col] - df["roll_mean_24"]) / (df["roll_std_24"] + 1e-5)
    df["zscore_168"] = (df[energy_col] - df["roll_mean_168"]) / (df["roll_std_168"] + 1e-5)

    # Drop NaN created by rolling/lag
    df = df.dropna()

    print("✅ Feature Engineering Completed")
    print("Total Features:", df.shape[1])
    print("🔎 Sample Columns:", df.columns[:50])

    return df